# 04c — `sec:twodigit` (Part 7 setup) across Pythia sizes

Companion to `04_arithmetic_main.ipynb`. **Part 7** of that notebook is the run that produced
the numbers reported in `sec:twodigit`: cell 83 there prints the paper's correlation sentence
verbatim (addition r = 0.316, subtraction r = 0.373, multiplication r = 0.605, division r = 0.440;
frequency r = -0.033 / -0.052 / -0.148 / -0.131). Part 7 is run only at 12B.

This notebook repeats **exactly that configuration** on every Pythia size, for the
smaller-models appendix:

* **inverse framing** via `op_cell`: `+: a+b`, `-: (a+b)-b = a`, `*: a*b`, `/: (a*b)/b = a`,
* operands `a, b in 1..100` — the full 100x100 grid, **10,000 cells per operation**
  (so 1- *and* 2-digit operands; division needs no exactness restriction, it is exact by construction),
* `problem_size = |c|` (the magnitude of the answer),
* frequency = **raw Pile count** of the displayed expression `x op y` (Part 7's convention,
  the one behind the r-values in the paper),
* 4-shot fixed prefix per operation (`kshot_fact`, seed 42),
* surprisal = total `-log P(answer | prompt)`; accuracy = greedy decode, exact match.

Note this is *not* the strictly-two-digit 90x90 grid of Part 4 (`a, b in 10..99`), despite the
section name — see `04b_arithmetic_2digit_model_sweep.ipynb` for that variant.

Outputs
* raw per-model/per-operation results -> `results/tables/part7_sweep_raw/*.csv` (checkpointed, so the sweep resumes)
* per-model heatmap figures -> `results/figures/figD_<model>.png` (+ split accuracy/surprisal panels)
* correlation-vs-scale figure -> `results/figures/part9b_inv_sweep_correlations.png`
* summary table -> `results/tables/part9b_inv_stats_all_models.csv` / `.docx`
* ready-to-paste LaTeX -> `results/tables/part9b_inv_appendix.tex`

In [1]:
# ── Environment (same cache layout as 04_arithmetic_main) ────────────────────
import os

ROOT = os.getcwd()
while ROOT != "/" and not os.path.isdir(os.path.join(ROOT, "hf", "hub")):
    ROOT = os.path.dirname(ROOT)
assert os.path.isdir(os.path.join(ROOT, "hf", "hub")), "could not locate <root>/hf/hub"

os.environ["HF_HOME"]          = os.path.join(ROOT, "hf")
os.environ["HF_HUB_CACHE"]     = os.path.join(ROOT, "hf", "hub")
os.environ["HF_DATASETS_CACHE"] = os.path.join(ROOT, "hf", "datasets")
os.environ.setdefault("HF_HUB_OFFLINE", "1")   # all 8 checkpoints are already cached
os.environ["PATH"] += ":/storage/home/hcoda1/9/kzhang430/.local/bin"

import re, json, gc, random, itertools, warnings
from collections import Counter

import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}  |  torch {torch.__version__}")

Using device: cpu  |  torch 2.7.0+cu126


In [2]:
# ── Configuration ────────────────────────────────────────────────────────────
K_SHOT     = 4       # matches Part 7
KSHOT_SEED = 42

BATCH_SIZE_LOGPROB  = 256
BATCH_SIZE_ACCURACY = 256

PYTHIA_MODELS = [
    "EleutherAI/pythia-70m-deduped",
    "EleutherAI/pythia-160m-deduped",
    "EleutherAI/pythia-410m-deduped",
    "EleutherAI/pythia-1b-deduped",
    "EleutherAI/pythia-1.4b-deduped",
    "EleutherAI/pythia-2.8b-deduped",
    "EleutherAI/pythia-6.9b-deduped",
    "EleutherAI/pythia-12b-deduped",
]

OPS       = ["+", "-", "*", "/"]
OP_NAMES  = {"+": "addition", "-": "subtraction", "*": "multiplication", "/": "division"}
OP_SYMBOL = {"+": "+", "-": "-", "*": "*", "/": "/"}
SHORT     = {"+": "add", "-": "sub", "*": "mul", "/": "div"}

FIG_DIR = "../results/figures"
TAB_DIR = "../results/tables"
RAW_DIR = f"{TAB_DIR}/part7_sweep_raw"
for d in (FIG_DIR, TAB_DIR, RAW_DIR):
    os.makedirs(d, exist_ok=True)

# file-name stem used in the appendix LaTeX, e.g. pythia-1.4b-deduped -> pythia1_4b
def fig_stem(mshort):
    s = mshort.replace("-deduped", "").replace("-", "").replace(".", "_")
    return f"figD_{s}"

print(f"k = {K_SHOT}, {len(PYTHIA_MODELS)} models, grid = 100x100 (operands 1..100, inverse framing)")

k = 4, 8 models, grid = 100x100 (operands 1..100, inverse framing)


In [3]:
# ── Utilities: model loading + evaluation (verbatim from 04_arithmetic_main) ──
def load_model(name, device=device):
    print(f"Loading {name}...")
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token_id is None:
        tok.pad_token_id = tok.eos_token_id
    tok.padding_side = "left"   # left-pad for correct batched generation (decoder-only)
    mdl = AutoModelForCausalLM.from_pretrained(
        name,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto" if device == "cuda" else None,
    )
    mdl.eval()
    n = sum(p.numel() for p in mdl.parameters()) / 1e6
    print(f"  Loaded: {n:.0f}M params on {next(mdl.parameters()).device}")
    return tok, mdl


@torch.inference_mode()
def eval_logprob(model, tokenizer, df, batch_size=32, prompt_col="prompt", target_col="target"):
    """Per-example log-probability of the target given the prompt."""
    prompts = df[prompt_col].tolist()
    targets = df[target_col].astype(str).tolist()
    full_texts = [f"{p} {t}" for p, t in zip(prompts, targets)]
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id

    out_rows = []
    for i in tqdm(range(0, len(prompts), batch_size), desc="logprob", leave=False):
        bp = prompts[i:i+batch_size]
        bf = full_texts[i:i+batch_size]
        prompt_enc = tokenizer(bp, add_special_tokens=False)
        full_enc   = tokenizer(bf, add_special_tokens=False)

        batch_ids, batch_mask, prompt_lens, target_lens = [], [], [], []
        for p_ids, full_ids in zip(prompt_enc["input_ids"], full_enc["input_ids"]):
            if full_ids[:len(p_ids)] != p_ids:
                raise ValueError("Prompt tokens are not a prefix of the full prompt+target tokens.")
            t_len = len(full_ids) - len(p_ids)
            if t_len <= 0:
                raise ValueError("Target tokenization produced an empty suffix.")
            batch_ids.append(full_ids)
            batch_mask.append([1]*len(full_ids))
            prompt_lens.append(len(p_ids))
            target_lens.append(t_len)

        max_len = max(len(x) for x in batch_ids)
        padded_ids  = [ids  + [pad_id]*(max_len-len(ids))  for ids  in batch_ids]
        padded_mask = [mask + [0]*(max_len-len(mask))      for mask in batch_mask]

        input_ids      = torch.tensor(padded_ids,  device=model.device)
        attention_mask = torch.tensor(padded_mask, device=model.device)
        logits    = model(input_ids=input_ids, attention_mask=attention_mask).logits
        log_probs = torch.log_softmax(logits[:, :-1, :], dim=-1)
        next_tok  = input_ids[:, 1:]
        token_lp  = log_probs.gather(2, next_tok.unsqueeze(-1)).squeeze(-1)

        for p_len, t_len, tlp in zip(prompt_lens, target_lens, token_lp):
            start = p_len - 1
            tgt_lp = tlp[start:start+t_len]
            out_rows.append({
                "target_logprob":     tgt_lp.sum().item(),
                "avg_target_logprob": tgt_lp.mean().item(),
                "n_target_tokens":    int(t_len),
            })

    result = df.copy()
    result["target_logprob"]     = [r["target_logprob"]     for r in out_rows]
    result["avg_target_logprob"] = [r["avg_target_logprob"] for r in out_rows]
    result["n_target_tokens"]    = [r["n_target_tokens"]    for r in out_rows]
    # surprisal = negative TOTAL (summed) target log-prob = -log P(answer | prompt).
    result["surprisal"] = -result["target_logprob"]
    return result


@torch.inference_mode()
def eval_accuracy(model, tokenizer, df, batch_size=16, prompt_col="prompt",
                  target_col="target", max_new_tokens=6):
    """Greedy decoding + exact-match accuracy."""
    prompts = df[prompt_col].tolist()
    targets = df[target_col].astype(str).tolist()
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id

    preds = []
    for i in tqdm(range(0, len(prompts), batch_size), desc="accuracy", leave=False):
        bp  = prompts[i:i+batch_size]
        enc = tokenizer(bp, return_tensors="pt", padding=True,
                        add_special_tokens=False).to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=pad_id)
        for inp_ids, out_ids in zip(enc["input_ids"], out):
            gen = tokenizer.decode(out_ids[len(inp_ids):], skip_special_tokens=True).strip()
            m = re.match(r"(-?\d+)", gen)
            preds.append(m.group(1) if m else gen)

    result = df.copy()
    result["prediction"] = preds
    result["correct"]    = [p.strip() == t.strip() for p, t in zip(preds, targets)]
    return result


def apply_kshot_prefix(df, prefix, prompt_col="prompt", out_col="eval_prompt"):
    df = df.copy()
    df[out_col] = (prefix + "\n" + df[prompt_col]) if prefix else df[prompt_col]
    return df


def safe_pearson(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    if np.std(x) == 0 or np.std(y) == 0 or len(x) < 3:
        return np.nan, np.nan
    r, p = pearsonr(x, y)
    return r, p

print("eval utilities ready")

eval utilities ready


In [4]:
# ── Pile frequency for all four operations (same as 04_arithmetic_main) ──────
JSONL_PATH = "../data/pile_arith_matches_full_fresh_deduped.jsonl"
_OP_CANON = {"+": "+", "-": "-", "*": "*", "×": "*", "/": "/", "÷": "/"}

def load_freq_expr(path=JSONL_PATH):
    freq = Counter()
    with open(path, "r") as f:
        for line in f:
            r = json.loads(line.strip())
            op = _OP_CANON.get(r.get("op"))
            if op is None:
                continue
            freq[(op, int(r["a"]), int(r["b"]))] += 1
    return freq

freq_expr = load_freq_expr()
print(f"Loaded {sum(freq_expr.values()):,} expressions across {len(freq_expr):,} unique (op, a, b) keys")

Loaded 204,073 expressions across 39,151 unique (op, a, b) keys


In [5]:
# ── Part-7 problem construction (inverse framing, operands 1..100) ─────────
def op_cell(op, a, b):
    """(x, y, answer) shown at grid cell (a, b).
    Inverse framing: subtraction undoes addition, division undoes multiplication."""
    if op == "+":  return a,     b, a + b      # a + b = a+b
    if op == "-":  return a + b, b, a          # (a+b) - b = a
    if op == "*":  return a,     b, a * b      # a * b = a*b
    if op == "/":  return a * b, b, a          # (a*b) / b = a
    raise ValueError(f"Unknown op: {op!r}")


def gen_inverse(op, pairs):
    """Part 7's problem generator: size = |c| (the answer), freq = RAW Pile count of 'x op y'."""
    sym = OP_SYMBOL[op]; rows = []
    for a, b in pairs:
        x, y, ans = op_cell(op, a, b)
        rows.append({"op": op, "a": a, "b": b, "x": x, "y": y,
                     "prompt": f"{x} {sym} {y} =", "target": str(ans), "answer": ans,
                     "problem_size": abs(ans),                  # size = |result|
                     "freq": freq_expr.get((op, x, y), 0)})     # freq of displayed 'x op y'
    return pd.DataFrame(rows)


def kshot_fact(op, pool_pairs, k=K_SHOT, seed=KSHOT_SEED):
    if k == 0:
        return ""
    rng = random.Random(seed)
    pp = list(pool_pairs); rng.shuffle(pp)
    sym = OP_SYMBOL[op]
    lines = []
    for a, b in pp[:k]:
        x, y, ans = op_cell(op, a, b)
        lines.append(f"{x} {sym} {y} = {ans}")
    return "\n".join(lines)


PAIRS_BOTH = [(a, b) for a in range(1, 101) for b in range(1, 101)]   # operands 1..100
problem_dfs_inv = {op: gen_inverse(op, PAIRS_BOTH) for op in OPS}

print(f"{len(PAIRS_BOTH)} cells per operation.  Example cell (a=37, b=48):")
for op in OPS:
    r = problem_dfs_inv[op].query("a == 37 and b == 48").iloc[0]
    print(f"  {OP_NAMES[op]:14s}: {r['prompt']} {r['target']:>5}   "
          f"size={r['problem_size']:>5}  pile_freq={r['freq']}")
print()
print("4-shot prefix for '+':")
print(kshot_fact("+", PAIRS_BOTH))

10000 cells per operation.  Example cell (a=37, b=48):
  addition      : 37 + 48 =    85   size=   85  pile_freq=2
  subtraction   : 85 - 48 =    37   size=   37  pile_freq=0
  multiplication: 37 * 48 =  1776   size= 1776  pile_freq=0
  division      : 1776 / 48 =    37   size=   37  pile_freq=0

4-shot prefix for '+':
38 + 72 = 110
67 + 73 = 140
73 + 62 = 135
8 + 61 = 69


In [6]:
# ── The sweep: Part-7 (inverse, 1..100) analysis on every Pythia size ───────
# Checkpointed: each (model, op) result is cached as a CSV, so an interrupted run resumes.
def raw_path(mshort, op):
    return f"{RAW_DIR}/{mshort}__{SHORT[op]}.csv"

RESULT_COLS = ["op", "a", "b", "x", "y", "answer", "problem_size", "freq",
               "surprisal", "target_logprob", "avg_target_logprob",
               "n_target_tokens", "prediction", "correct"]

sweep2_results = {}
for name in PYTHIA_MODELS:
    mshort = name.split("/")[-1]
    if all(os.path.exists(raw_path(mshort, op)) for op in OPS):
        sweep2_results[mshort] = {op: pd.read_csv(raw_path(mshort, op)) for op in OPS}
        print(f"[cached] {mshort}")
        continue

    tok_s, mdl_s = load_model(name)
    by_op = {}
    for op in OPS:
        p = raw_path(mshort, op)
        if os.path.exists(p):
            by_op[op] = pd.read_csv(p)
            print(f"  [cached] {mshort} {OP_NAMES[op]}")
            continue
        df = apply_kshot_prefix(problem_dfs_inv[op], kshot_fact(op, PAIRS_BOTH, k=K_SHOT))
        df = eval_logprob(mdl_s, tok_s, df, batch_size=BATCH_SIZE_LOGPROB, prompt_col="eval_prompt")
        df = eval_accuracy(mdl_s, tok_s, df, batch_size=BATCH_SIZE_ACCURACY, prompt_col="eval_prompt")
        df[RESULT_COLS].to_csv(p, index=False)
        by_op[op] = df
        print(f"  {mshort} | {OP_NAMES[op]:14s} | n={len(df)} | "
              f"acc={df['correct'].mean():.3f} | mean surprisal={df['surprisal'].mean():.3f}", flush=True)
    sweep2_results[mshort] = by_op

    del mdl_s, tok_s
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\nDone: {len(sweep2_results)} models x {len(OPS)} operations.")

[cached] pythia-70m-deduped
[cached] pythia-160m-deduped
[cached] pythia-410m-deduped
[cached] pythia-1b-deduped
[cached] pythia-1.4b-deduped
[cached] pythia-2.8b-deduped
[cached] pythia-6.9b-deduped
[cached] pythia-12b-deduped

Done: 8 models x 4 operations.


In [7]:
# ── Per-model, per-operation statistics ─────────────────────────────────────
# Columns mirror the single-digit appendix table (Part 5 / Part 8):
#   mean accuracy, SD, r(surprisal, problem size) + p, r(surprisal, frequency) + p.
# Part 7 (and hence sec:twodigit) correlates surprisal with the RAW Pile count, so that is
# the primary frequency column here; log1p(freq) is carried alongside for reference.
stat_rows = []
for mshort, by_op in sweep2_results.items():
    for op in OPS:
        d = by_op[op]
        acc  = d["correct"].astype(float).values
        surp = d["surprisal"].values.astype(float)
        size = d["problem_size"].values.astype(float)
        raw  = d["freq"].values.astype(float)
        logf = np.log1p(raw)
        r_size, p_size = safe_pearson(surp, size)
        r_lf,   p_lf   = safe_pearson(surp, logf)
        r_rf,   p_rf   = safe_pearson(surp, raw)
        stat_rows.append({
            "model": mshort, "operation": OP_NAMES[op].capitalize(), "n": len(d),
            "mean_acc": acc.mean(), "sd_acc": acc.std(),
            "r_surp_size": r_size, "p_size": p_size,
            "r_surp_logfreq": r_lf, "p_logfreq": p_lf,
            "r_surp_rawfreq": r_rf, "p_rawfreq": p_rf,
            "mean_surprisal": surp.mean(),
        })

stats2 = pd.DataFrame(stat_rows)
order = [m.split("/")[-1] for m in PYTHIA_MODELS]
stats2["model"] = pd.Categorical(stats2["model"], categories=order, ordered=True)
stats2 = stats2.sort_values(["model", "operation"]).reset_index(drop=True)
stats2.to_csv(f"{TAB_DIR}/part9b_inv_stats_all_models.csv", index=False)
print(f"Saved {TAB_DIR}/part9b_inv_stats_all_models.csv")
with pd.option_context("display.width", 200, "display.max_rows", 100):
    print(stats2.to_string(index=False, float_format=lambda v: f"{v: .4g}"))
stats2

Saved ../results/tables/part9b_inv_stats_all_models.csv
              model      operation     n  mean_acc    sd_acc  r_surp_size      p_size  r_surp_logfreq   p_logfreq  r_surp_rawfreq   p_rawfreq  mean_surprisal
 pythia-70m-deduped       Addition 10000    0.0067   0.08158        0.538           0         -0.1257   1.607e-36       -0.006416      0.5212           5.353
 pythia-70m-deduped       Division 10000    0.0108    0.1034       0.4841           0         -0.1668   2.724e-63         -0.1057   2.971e-26           4.915
 pythia-70m-deduped Multiplication 10000    0.0001  0.009999       0.6443           0         -0.5745           0         -0.2916  3.502e-195           9.291
 pythia-70m-deduped    Subtraction 10000         0         0       0.7506           0         -0.2912  1.067e-194         -0.1075   4.272e-27           5.255
pythia-160m-deduped       Addition 10000    0.0083   0.09073       0.2513  7.085e-144          0.1021   1.345e-24         0.01127      0.2598           5.

,model,operation,n,mean_acc,sd_acc,r_surp_size,p_size,r_surp_logfreq,p_logfreq,r_surp_rawfreq,p_rawfreq,mean_surprisal
0,pythia-70m-deduped,Addition,10000,0.0067,0.081579,0.537956,0.000000e+00,-0.125730,1.606969e-36,-0.006416,5.211985e-01,5.353058
1,pythia-70m-deduped,Division,10000,0.0108,0.103360,0.484096,0.000000e+00,-0.166778,2.724228e-63,-0.105719,2.971198e-26,4.915267
2,pythia-70m-deduped,Multiplication,10000,0.0001,0.009999,0.644307,0.000000e+00,-0.574461,0.000000e+00,-0.291574,3.502340e-195,9.291058
3,pythia-70m-deduped,Subtraction,10000,0.0000,0.000000,0.750630,0.000000e+00,-0.291225,1.066692e-194,-0.107502,4.272119e-27,5.254948
4,pythia-160m-deduped,Addition,10000,0.0083,0.090725,0.251322,7.085021e-144,0.102121,1.345197e-24,0.011270,2.597686e-01,5.155866
5,pythia-160m-deduped,Division,10000,0.0106,0.102409,0.700005,0.000000e+00,-0.265657,3.835304e-161,-0.171300,1.016878e-66,5.064642
6,pythia-160m-deduped,Multiplication,10000,0.0004,0.019996,0.688827,0.000000e+00,-0.615492,0.000000e+00,-0.306447,2.065576e-216,8.719520
7,pythia-160m-deduped,Subtraction,10000,0.0004,0.019996,0.754349,0.000000e+00,-0.271292,3.129273e-168,-0.124651,6.392293e-36,5.454325
8,pythia-410m-deduped,Addition,10000,0.0005,0.022355,0.694086,0.000000e+00,-0.436985,0.000000e+00,-0.091485,4.883865e-20,4.651979
9,pythia-410m-deduped,Division,10000,0.0092,0.095474,0.675573,0.000000e+00,-0.291008,2.125122e-194,-0.201862,1.893700e-92,4.882438


In [8]:
# ── Verification: reproduce the sec:twodigit paragraph from the 12B row ─────
# Part 7 cell 83 of 04_arithmetic_main.ipynb printed this sentence; the published values are
#   size: add 0.316, sub 0.373, mul 0.605, div 0.440
#   freq: add -0.033, sub -0.052, mul -0.148, div -0.131
PUBLISHED = {"size": {"+": 0.316, "-": 0.373, "*": 0.605, "/": 0.440},
             "freq": {"+": -0.033, "-": -0.052, "*": -0.148, "/": -0.131}}

twelve = "pythia-12b-deduped"
if twelve in sweep2_results:
    sub = stats2[stats2["model"] == twelve].set_index("operation")
    rp = {}
    for op in OPS:
        r = sub.loc[OP_NAMES[op].capitalize()]
        rp[op] = {"size": (r["r_surp_size"], r["p_size"]),
                  "freq": (r["r_surp_rawfreq"], r["p_rawfreq"])}

    paragraph = (
        "This is shown quantitatively in the high correlation between surprisal and "
        "problem size (c) for each operation: "
        "addition (r = {:.3f}, p = {:.3g}), subtraction (r = {:.3f}, p = {:.3g}), "
        "multiplication (r = {:.3f}, p = {:.3g}), and division (r = {:.3f}, p = {:.3g}). "
        "Finally, we evaluated whether surprisal on a problem was correlated with the "
        "frequency of that problem in the Pile. This was the case: "
        "addition (r = {:.3f}, p = {:.3g}), subtraction ({:.3f}, p = {:.3g}), "
        "multiplication ({:.3f}, p = {:.3g}), and division ({:.3f}, p = {:.3g})."
    ).format(
        *rp["+"]["size"], *rp["-"]["size"], *rp["*"]["size"], *rp["/"]["size"],
        *rp["+"]["freq"], *rp["-"]["freq"], *rp["*"]["freq"], *rp["/"]["freq"],
    )
    print(paragraph)
    print()
    print(f"{'':14s} {'r_size':>9s} {'published':>10s} {'delta':>8s}   "
          f"{'r_freq':>9s} {'published':>10s} {'delta':>8s}")
    for op in OPS:
        rs = rp[op]["size"][0]; rf = rp[op]["freq"][0]
        ps = PUBLISHED["size"][op]; pf = PUBLISHED["freq"][op]
        print(f"{OP_NAMES[op]:14s} {rs:9.3f} {ps:10.3f} {rs-ps:+8.3f}   "
              f"{rf:9.3f} {pf:10.3f} {rf-pf:+8.3f}")
    print()
    print("Accuracies (this run) vs. the values reported in sec:twodigit:")
    PUB_ACC = {"+": 0.831, "-": 0.796, "*": 0.388, "/": 0.509}
    for op in OPS:
        a = sub.loc[OP_NAMES[op].capitalize(), "mean_acc"]
        print(f"  {OP_NAMES[op]:14s} {a:6.3f}  vs paper {PUB_ACC[op]:.3f}   delta {a-PUB_ACC[op]:+.3f}")
    print("\nNOTE: the accuracy quadruple reported in sec:twodigit (0.831/0.796/0.388/0.509) does")
    print("not match any run recorded in this repo. Part 7's own accuracy cell produced")
    print("0.861/0.806/0.382/0.370. The correlations, by contrast, match Part 7 exactly.")

This is shown quantitatively in the high correlation between surprisal and problem size (c) for each operation: addition (r = 0.316, p = 9.97e-231), subtraction (r = 0.373, p = 0), multiplication (r = 0.605, p = 0), and division (r = 0.440, p = 0). Finally, we evaluated whether surprisal on a problem was correlated with the frequency of that problem in the Pile. This was the case: addition (r = -0.033, p = 0.00082), subtraction (-0.052, p = 1.52e-07), multiplication (-0.148, p = 3.88e-50), and division (-0.131, p = 2.26e-39).

                  r_size  published    delta      r_freq  published    delta
addition           0.316      0.316   +0.000      -0.033     -0.033   -0.000
subtraction        0.373      0.373   -0.000      -0.052     -0.052   -0.000
multiplication     0.605      0.605   -0.000      -0.148     -0.148   -0.000
division           0.440      0.440   +0.000      -0.131     -0.131   +0.000

Accuracies (this run) vs. the values reported in sec:twodigit:
  addition        

In [9]:
# ── Per-model surprisal heatmaps (2x2, same style as the single-digit figures) ──
# Mirrors Part 8 cell 92 of 04_arithmetic_main.ipynb: one panel per operation, a shared
# colour scale (0 .. 95th percentile of the pooled surprisal), per-panel colourbar.
tick_pos = list(range(0, 100, 20)); tick_lab = list(range(1, 101, 20))

for mshort, by_op in sweep2_results.items():
    pooled = np.concatenate([by_op[op]["surprisal"].values for op in OPS])
    vmax = float(np.quantile(pooled, 0.95))
    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    for ax, op in zip(axes.ravel(), OPS):
        piv = (by_op[op].pivot(index="a", columns="b", values="surprisal")
               .reindex(index=range(1, 101), columns=range(1, 101)))
        im = ax.imshow(piv.values, cmap="RdYlGn_r", vmin=0, vmax=vmax,
                       aspect="auto", origin="lower")
        ax.set_xticks(tick_pos); ax.set_yticks(tick_pos)
        ax.set_xticklabels(tick_lab); ax.set_yticklabels(tick_lab)
        ax.set_xlabel("b"); ax.set_ylabel("a")
        ax.set_title(f"{OP_NAMES[op].capitalize()} (mean {by_op[op]['surprisal'].mean():.2f})")
        plt.colorbar(im, ax=ax, label="surprisal")
    fig.suptitle(f"{mshort} (operands 1-100, k={K_SHOT})", y=1.0, fontsize=12)
    plt.tight_layout()
    out = f"{FIG_DIR}/{fig_stem(mshort)}.png"
    plt.savefig(out, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print("saved", out)

saved ../results/figures/figD_pythia70m.png
saved ../results/figures/figD_pythia160m.png
saved ../results/figures/figD_pythia410m.png
saved ../results/figures/figD_pythia1b.png
saved ../results/figures/figD_pythia1_4b.png
saved ../results/figures/figD_pythia2_8b.png
saved ../results/figures/figD_pythia6_9b.png
saved ../results/figures/figD_pythia12b.png


In [10]:
# ── Paper Figure 4 (12B, operands 1-100) ────────────────────────────────────
# Same visual style as the existing Figure 4 (Part-6/7 heatmaps): per-panel colourbar
# labelled "surprisal (nats)", titles "Operation (mean surprisal = X.XXX)", shared colour
# scale (0 .. 95th percentile of pooled surprisal), no suptitle — the LaTeX caption carries it.
FIG4_MODEL = "pythia-12b-deduped"
by_op4 = sweep2_results[FIG4_MODEL]

tick_pos4 = [0, 19, 39, 59, 79, 99]; tick_lab4 = [1, 20, 40, 60, 80, 100]
pooled4 = np.concatenate([by_op4[op]["surprisal"].values for op in OPS])
vmax4 = float(np.quantile(pooled4, 0.95))

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for ax, op in zip(axes.ravel(), OPS):
    d = by_op4[op]
    piv = (d.pivot(index="a", columns="b", values="surprisal")
           .reindex(index=range(1, 101), columns=range(1, 101)))
    im = ax.imshow(piv.values, cmap="RdYlGn_r", vmin=0, vmax=vmax4,
                   aspect="auto", origin="lower")
    ax.set_xticks(tick_pos4); ax.set_yticks(tick_pos4)
    ax.set_xticklabels(tick_lab4); ax.set_yticklabels(tick_lab4)
    ax.set_xlabel("b"); ax.set_ylabel("a")
    ax.set_title(f"{OP_NAMES[op].capitalize()} (mean surprisal = {d['surprisal'].mean():.3f})")
    plt.colorbar(im, ax=ax, label="surprisal (nats)")
plt.tight_layout()
OUT4 = f"{FIG_DIR}/fig4_surprisal_1to100_{FIG4_MODEL}.png"
plt.savefig(OUT4, dpi=150, bbox_inches="tight")
plt.close(fig)
print("saved", OUT4)
print()
print("Figure 4 mean surprisals on the 1-100 grid (replacing the 1-99 values):")
for op in OPS:
    print(f"  {OP_NAMES[op]:14s} {by_op4[op]['surprisal'].mean():.3f}")

saved ../results/figures/fig4_surprisal_1to100_pythia-12b-deduped.png

Figure 4 mean surprisals on the 1-100 grid (replacing the 1-99 values):
  addition       0.494
  subtraction    0.624
  multiplication 3.500
  division       1.940


In [11]:
# ── Correlations vs. model size ─────────────────────────────────────────────
piv_size = stats2.pivot(index="model", columns="operation", values="r_surp_size").reindex(order)
piv_freq = stats2.pivot(index="model", columns="operation", values="r_surp_rawfreq").reindex(order)
piv_acc  = stats2.pivot(index="model", columns="operation", values="mean_acc").reindex(order)

PARAMS_M = {"pythia-70m-deduped": 70.4, "pythia-160m-deduped": 162, "pythia-410m-deduped": 405,
            "pythia-1b-deduped": 1011, "pythia-1.4b-deduped": 1414, "pythia-2.8b-deduped": 2775,
            "pythia-6.9b-deduped": 6857, "pythia-12b-deduped": 11846}
x = np.array([PARAMS_M[m] for m in piv_size.index])
colors = {"Addition": "tab:blue", "Subtraction": "tab:orange",
          "Multiplication": "tab:green", "Division": "tab:red"}

fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for opname, c in colors.items():
    axes[0].plot(x, piv_acc[opname],  "o-", color=c, label=opname)
    axes[1].plot(x, piv_size[opname], "o-", color=c, label=opname)
    axes[2].plot(x, piv_freq[opname], "o-", color=c, label=opname)
for ax, t, yl in [(axes[0], "accuracy", "mean accuracy"),
                  (axes[1], "surprisal ~ problem size", "Pearson r"),
                  (axes[2], "surprisal ~ raw Pile frequency", "Pearson r")]:
    ax.set_xscale("log"); ax.axhline(0, color="0.7", lw=0.8)
    ax.set_xlabel("parameters (M)"); ax.set_ylabel(yl); ax.set_title(t)
    ax.legend(title="operation", fontsize=8)
fig.suptitle(f"sec:twodigit setup across Pythia sizes (inverse framing, k={K_SHOT}, operands 1-100)", y=1.03, fontsize=14)
plt.tight_layout()
out = f"{FIG_DIR}/part9b_inv_sweep_correlations.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
print("saved", out)
plt.show()

saved ../results/figures/part9b_inv_sweep_correlations.png


In [12]:
# ── LaTeX: appendix section in the same \smallmodeltable format ─────────────
PRETTY = {"pythia-70m-deduped": "Pythia-70M-deduped", "pythia-160m-deduped": "Pythia-160M-deduped",
          "pythia-410m-deduped": "Pythia-410M-deduped", "pythia-1b-deduped": "Pythia-1B-deduped",
          "pythia-1.4b-deduped": "Pythia-1.4B-deduped", "pythia-2.8b-deduped": "Pythia-2.8B-deduped",
          "pythia-6.9b-deduped": "Pythia-6.9B-deduped", "pythia-12b-deduped": "Pythia-12B-deduped"}
LABEL  = {"pythia-70m-deduped": "70minv", "pythia-160m-deduped": "160minv", "pythia-410m-deduped": "410minv",
          "pythia-1b-deduped": "1binv", "pythia-1.4b-deduped": "14binv", "pythia-2.8b-deduped": "28binv",
          "pythia-6.9b-deduped": "69binv", "pythia-12b-deduped": "12binv"}
OPROWS = ["Addition", "Subtraction", "Multiplication", "Division"]

def f3(v):  return "n/a" if v != v else f"{v:.3f}"

def fg(v):
    """p-value formatter. With n=8100 many p-values underflow to exactly 0.0 in
    double precision, so report those as a bound rather than a bare '0'."""
    if v != v:
        return "n/a"
    if v == 0.0:
        return r"$<$1e-308"
    return f"{v:.3g}"

def rows_for(mshort, freq_kind="rawfreq"):
    sub = stats2[stats2["model"] == mshort].set_index("operation")
    cells = []
    for opn in OPROWS:
        r = sub.loc[opn]
        cells.append([opn, f3(r["mean_acc"]), f3(r["sd_acc"]),
                      f3(r["r_surp_size"]), fg(r["p_size"]),
                      f3(r[f"r_surp_{freq_kind}"]), fg(r[f"p_{freq_kind}"])])
    w = [max(len(c[i]) for c in cells) for i in range(7)]
    lines = []
    for c in cells:
        lines.append("    {:<{}} & {:>{}} & {:>{}} & ${:>{}}$ & {:<{}} & ${:>{}}$ & {:<{}} \\\\".format(
            c[0], w[0], c[1], w[1], c[2], w[2], c[3], w[3], c[4], w[4], c[5], w[5], c[6], w[6]))
    return "\n".join(lines)


def build_tex(freq_kind, freq_desc, models):
    L = []
    L.append(r"\section{4-shot arithmetic on smaller models: 1- and 2-digit operands}")
    L.append(r"\label{app:smaller-models-twodigit}")
    L.append("")
    L.append(r"What follows are results for smaller Pythia models on the same experiment as")
    L.append(r"Section~\ref{sec:twodigit}: the inverse framing ($+: a+b$, $-: (a+b)-b$, $*: a\times b$,")
    L.append(r"$/: (a\times b)/b$) over the full $100\times100$ grid of operands $a,b \in [1,100]$, 4-shot,")
    L.append(r"including both heatmaps and tables. Problem size is $|c|$, the magnitude of the answer;")
    L.append(r"the frequency column correlates")
    L.append(freq_desc)
    L.append("")
    L.append(r"\newcommand{\smallmodeltabletwod}[3]{%")
    L.append(r"\begin{table}[!htb]")
    L.append(r"  \caption{#1}")
    L.append(r"  \label{#2}")
    L.append(r"  \centering")
    L.append(r"  \begin{tabular}{lrrrrrr}")
    L.append(r"    \toprule")
    L.append(r"    & \multicolumn{2}{c}{Accuracy}")
    L.append(r"    & \multicolumn{2}{c}{Surprisal--size}")
    L.append(r"    & \multicolumn{2}{c}{Surprisal--frequency} \\")
    L.append(r"    \cmidrule(lr){2-3}\cmidrule(lr){4-5}\cmidrule(lr){6-7}")
    L.append(r"    Operation & $M$ & $SD$ & $r$ & $p$ & $r$ & $p$ \\")
    L.append(r"    \midrule")
    L.append(r"    #3")
    L.append(r"    \bottomrule")
    L.append(r"  \end{tabular}")
    L.append(r"\end{table}}")
    L.append("")
    for i, mshort in enumerate(models):
        pretty = PRETTY[mshort]
        L.append(r"\subsection{%s}" % pretty)
        L.append("")
        L.append(r"\smallmodeltabletwod{%s, 4-shot, operands $1$--$100$.}{tab:%s}{%%"
                 % (pretty, LABEL[mshort]))
        L.append(rows_for(mshort, freq_kind))
        L.append("}")
        L.append("")
        L.append(r"\begin{figure}[!htb]")
        L.append(r"  \centering")
        L.append(r"  \includegraphics[width=0.85\linewidth]{%s.png}" % fig_stem(mshort))
        L.append(r"  \caption{%s, operands $1$--$100$, $k=4$.}" % pretty)
        L.append(r"\end{figure}")
        L.append(r"\FloatBarrier")
        if i < len(models) - 1:
            L.append(r"\clearpage")
        L.append("")
    L.append(r"\FloatBarrier")
    return "\n".join(L)


smaller = [m for m in order if m in stats2["model"].values and m != "pythia-12b-deduped"]

# Primary appendix uses the RAW Pile count, matching Part 7 / sec:twodigit.
tex_raw = build_tex("rawfreq",
                    r"surprisal with the raw Pile count of the displayed expression.",
                    smaller)
tex_log = build_tex("logfreq",
                    r"surprisal with $\log(1+\text{Pile count})$ of the displayed expression.",
                    smaller)

with open(f"{TAB_DIR}/part9b_inv_appendix.tex", "w") as f:
    f.write(tex_raw)
with open(f"{TAB_DIR}/part9b_inv_appendix_logfreq.tex", "w") as f:
    f.write(tex_log)

# all-model version (includes 12B) for reference
with open(f"{TAB_DIR}/part9b_inv_appendix_allmodels.tex", "w") as f:
    f.write(build_tex("rawfreq",
                      r"surprisal with the raw Pile count of the displayed expression.",
                      [m for m in order if m in stats2["model"].values]))

print(f"Saved {TAB_DIR}/part9b_inv_appendix.tex (raw freq, matches sec:twodigit),")
print(f"      {TAB_DIR}/part9b_inv_appendix_logfreq.tex (log1p freq),")
print(f"      {TAB_DIR}/part9b_inv_appendix_allmodels.tex")
print()
print(tex_raw)

Saved ../results/tables/part9b_inv_appendix.tex (raw freq, matches sec:twodigit),
      ../results/tables/part9b_inv_appendix_logfreq.tex (log1p freq),
      ../results/tables/part9b_inv_appendix_allmodels.tex

\section{4-shot arithmetic on smaller models: 1- and 2-digit operands}
\label{app:smaller-models-twodigit}

What follows are results for smaller Pythia models on the same experiment as
Section~\ref{sec:twodigit}: the inverse framing ($+: a+b$, $-: (a+b)-b$, $*: a\times b$,
$/: (a\times b)/b$) over the full $100\times100$ grid of operands $a,b \in [1,100]$, 4-shot,
including both heatmaps and tables. Problem size is $|c|$, the magnitude of the answer;
the frequency column correlates
surprisal with the raw Pile count of the displayed expression.

\newcommand{\smallmodeltabletwod}[3]{%
\begin{table}[!htb]
  \caption{#1}
  \label{#2}
  \centering
  \begin{tabular}{lrrrrrr}
    \toprule
    & \multicolumn{2}{c}{Accuracy}
    & \multicolumn{2}{c}{Surprisal--size}
    & \multicolumn{2}

In [13]:
# ── .docx export (one table per model), mirroring part8_stats_all_models.docx ─
try:
    from docx import Document
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-docx"])
    from docx import Document

# plain-text p formatter for Word (no LaTeX math around the "<")
def fgd(v):
    if v != v:
        return "n/a"
    if v == 0.0:
        return "<1e-308"
    return f"{v:.3g}"

cols = ["Operation", "Mean accuracy", "SD", "r (surprisal-size)", "p",
        "r (surprisal-raw freq)", "p", "r (surprisal-log freq)", "p"]

doc = Document()
doc.add_heading("sec:twodigit setup (inverse framing, operands 1-100, 4-shot): "
                "per-operation statistics by model", level=1)
for mshort in order:
    if mshort not in stats2["model"].values:
        continue
    sub = stats2[stats2["model"] == mshort].set_index("operation")
    doc.add_heading(PRETTY[mshort], level=2)
    t = doc.add_table(rows=1, cols=len(cols)); t.style = "Table Grid"
    for i, c in enumerate(cols):
        t.rows[0].cells[i].text = c
    for opn in OPROWS:
        r = sub.loc[opn]
        vals = [opn, f3(r["mean_acc"]), f3(r["sd_acc"]),
                f3(r["r_surp_size"]), fgd(r["p_size"]),
                f3(r["r_surp_rawfreq"]), fgd(r["p_rawfreq"]),
                f3(r["r_surp_logfreq"]), fgd(r["p_logfreq"])]
        cells = t.add_row().cells
        for i, v in enumerate(vals):
            cells[i].text = v
    doc.add_paragraph("")

OUT = f"{TAB_DIR}/part9b_inv_stats_all_models.docx"
doc.save(OUT)
print("Saved", OUT)

Saved ../results/tables/part9b_inv_stats_all_models.docx
